In [14]:
# =========================
# 1. LOAD EVERYTHING
# =========================
import joblib
import numpy as np

model = joblib.load('model.pkl')
mlb = joblib.load('mlb.pkl')
le = joblib.load('le.pkl')
weights = joblib.load('weights.pkl')
desc_dict = joblib.load('desc.pkl')
prec_dict = joblib.load('prec.pkl')

# Define severe diseases (cleaned properly)
severe_diseases = {
    'AIDS', 'Cervical cancer', 'Diabetes',
    'Fungal infection', 'Osteoarthritis', 'Varicose veins'
}


# =========================
# 2. CLEAN INPUT
# =========================
def normalize(text):
    return str(text).strip().lower().replace(" ", "_")


# =========================
# 3. MAIN PREDICTION FUNCTION
# =========================
def predict_disease(symptoms):

    # ---- STEP 1: CLEAN INPUT ----
    symptoms = [normalize(s) for s in symptoms if str(s).strip() != ""]

    if len(symptoms) == 0:
        return {"error": "No symptoms provided"}

    # ---- STEP 2: VALIDATION ----
    valid_symptoms = set(mlb.classes_)

    valid = [s for s in symptoms if s in valid_symptoms]
    invalid = [s for s in symptoms if s not in valid_symptoms]

    if len(valid) == 0:
      return {"error": f"No valid symptoms. Invalid: {invalid}"}

    # Remove duplicates
    cleaned = list(set(valid))

    if len(cleaned) < 2:
      return {"error": "Provide at least 2 valid symptoms"}


    # ---- STEP 3: ENCODING ----
    vec = mlb.transform([cleaned])[0]

    if np.sum(vec) == 0:
        return {"error": "Symptoms not sufficient for prediction"}

    vec_weighted = vec.astype(float) * weights

    # ---- STEP 4: MODEL PREDICTION ----
    probs = model.predict_proba([vec_weighted])[0]
    max_prob = np.max(probs)

    top3_idx = np.argsort(probs)[-3:][::-1]
    top3 = [(le.classes_[i], round(float(probs[i]), 3)) for i in top3_idx]

    top_disease = top3[0][0]

    # ---- STEP 5: CONFIDENCE CONTROL ----
    warning_msg = None
    if invalid:
      warning_msg = f"Ignored Invalid symptoms: {invalid}"

    if max_prob < 0.4:
        return {
            "warning": "Low confidence prediction",
            "top3": top3,
            "note": warning_msg
        }

    # ---- STEP 6: ALERT LOGIC ----
    alert = "Consult a professional for confirmation."

    if top_disease in severe_diseases and max_prob > 0.6:
        alert = "🚨 See a doctor immediately!"

    # ---- STEP 7: FINAL RESPONSE ----
    return {
        "top3": top3,
        "description": desc_dict.get(top_disease, "N/A"),
        "precautions": prec_dict.get(top_disease, []),
        "confidence": round(float(max_prob), 3),
        "alert": alert,
        "note": warning_msg
    }


# =========================
# 4. TEST CASES
# =========================
if __name__ == "__main__":
    print(predict_disease(['shivering', 'chills', 'watering_from_eyes']))
    print(predict_disease([]))
    print(predict_disease(['']))
    print(predict_disease(['random']))
    print(predict_disease(['itching']))
    print(predict_disease(['itching', 'random']))
    print(predict_disease(['itching', 'skin_rash']))

{'top3': [('Allergy', 0.992), ('Malaria', 0.0), ('Typhoid', 0.0)], 'description': "An allergy is an immune system response to a foreign substance that's not typically harmful to your body.They can include certain foods, pollen, or pet dander. Your immune system's job is to keep you healthy by fighting harmful pathogens.", 'precautions': ['apply calamine', 'cover area with bandage', 'use ice to compress itching'], 'confidence': 0.992, 'alert': 'Consult a professional for confirmation.', 'note': None}
{'error': 'No symptoms provided'}
{'error': 'No symptoms provided'}
{'error': "No valid symptoms. Invalid: ['random']"}
{'error': 'Provide at least 2 valid symptoms'}
{'error': 'Provide at least 2 valid symptoms'}
{'warning': 'Low confidence prediction', 'top3': [('Fungal infection', 0.069), ('Acne', 0.05), ('Impetigo', 0.044)], 'note': None}
